In [54]:
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [55]:
df = pd.read_csv("../data/application_train.csv")

print("Application data loaded!")
print("Shape:", df.shape)

Application data loaded!
Shape: (307511, 122)


In [56]:
df = df.copy()

df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25

df["EMPLOYMENT_YEARS"] = (
    -df["DAYS_EMPLOYED"] / 365.25
)

df["CREDIT_INCOME_RATIO"] = (
    df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
)

df["ANNUITY_INCOME_RATIO"] = (
    df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
)

df["CREDIT_GOODS_RATIO"] = (
    df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"]
)

print("Application features created!")
print("Shape:", df.shape)

Application features created!
Shape: (307511, 127)


## Historical Credit Feature Engineering

The Home Credit dataset contains several historical sources that provide information about an applicant's previous borrowing and repayment behavior.

The following datasets will be aggregated at the applicant level using `SK_ID_CURR`:

- Bureau credit history
- Bureau monthly balance history
- Previous loan applications
- Installment payment history
- POS cash-loan history
- Credit-card history

The objective is to transform these transaction-level and account-level records into meaningful applicant-level risk features.

In [57]:
bureau = pd.read_csv("../data/bureau.csv")

print("Bureau shape:", bureau.shape)
print("Unique applicants:", bureau["SK_ID_CURR"].nunique())

Bureau shape: (1716428, 17)
Unique applicants: 305811


In [58]:
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_LOAN_COUNT=("SK_ID_BUREAU", "count"),
    
    BUREAU_ACTIVE_COUNT=(
        "CREDIT_ACTIVE",
        lambda x: (x == "Active").sum()
    ),
    
    BUREAU_CLOSED_COUNT=(
        "CREDIT_ACTIVE",
        lambda x: (x == "Closed").sum()
    ),
    
    BUREAU_CREDIT_SUM=("AMT_CREDIT_SUM", "sum"),
    BUREAU_CREDIT_MEAN=("AMT_CREDIT_SUM", "mean"),
    BUREAU_CREDIT_MAX=("AMT_CREDIT_SUM", "max"),
    
    BUREAU_DEBT_SUM=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_DEBT_MEAN=("AMT_CREDIT_SUM_DEBT", "mean"),
    
    BUREAU_OVERDUE_SUM=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    BUREAU_OVERDUE_MAX=("AMT_CREDIT_SUM_OVERDUE", "max"),
    
    BUREAU_DPD_MEAN=("CREDIT_DAY_OVERDUE", "mean"),
    BUREAU_DPD_MAX=("CREDIT_DAY_OVERDUE", "max"),
    
    BUREAU_PROLONG_COUNT=("CNT_CREDIT_PROLONG", "sum"),
    
    BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
    BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
    
    BUREAU_CREDIT_TYPE_COUNT=("CREDIT_TYPE", "nunique")
).reset_index()

print("Bureau features:", bureau_agg.shape)
bureau_agg.head()

Bureau features: (305811, 17)


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_CLOSED_COUNT,BUREAU_CREDIT_SUM,BUREAU_CREDIT_MEAN,BUREAU_CREDIT_MAX,BUREAU_DEBT_SUM,BUREAU_DEBT_MEAN,BUREAU_OVERDUE_SUM,BUREAU_OVERDUE_MAX,BUREAU_DPD_MEAN,BUREAU_DPD_MAX,BUREAU_PROLONG_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MIN,BUREAU_CREDIT_TYPE_COUNT
0,100001,7,3,4,1453365.000,207623.571429,378000.0,596686.5,85240.928571,0.0,0.0,0.0,0,0,-735.000000,-1572,1
1,100002,8,2,6,865055.565,108131.945625,450000.0,245781.0,49156.200000,0.0,0.0,0.0,0,0,-874.000000,-1437,2
2,100003,4,1,3,1017400.500,254350.125000,810000.0,0.0,0.000000,0.0,0.0,0.0,0,0,-1400.750000,-2586,2
3,100004,2,0,2,189037.800,94518.900000,94537.8,0.0,0.000000,0.0,0.0,0.0,0,0,-867.000000,-1326,1
4,100005,3,2,1,657126.000,219042.000000,568800.0,568408.5,189469.500000,0.0,0.0,0.0,0,0,-190.666667,-373,2


In [59]:
bureau_agg["BUREAU_DEBT_CREDIT_RATIO"] = (
    bureau_agg["BUREAU_DEBT_SUM"] /
    bureau_agg["BUREAU_CREDIT_SUM"].replace(0, np.nan)
)

bureau_agg["BUREAU_ACTIVE_RATIO"] = (
    bureau_agg["BUREAU_ACTIVE_COUNT"] /
    bureau_agg["BUREAU_LOAN_COUNT"]
)

bureau_agg["BUREAU_OVERDUE_LOAN_RATIO"] = (
    bureau_agg["BUREAU_OVERDUE_SUM"] /
    bureau_agg["BUREAU_CREDIT_SUM"].replace(0, np.nan)
)

print("Bureau ratio features added!")

Bureau ratio features added!


In [60]:
df = df.merge(
    bureau_agg,
    on="SK_ID_CURR",
    how="left"
)

print("After Bureau merge:", df.shape)

After Bureau merge: (307511, 146)


In [61]:
bureau_balance = pd.read_csv(
    "../data/bureau_balance.csv"
)

print("Bureau Balance shape:", bureau_balance.shape)
print(
    "Unique bureau loans:",
    bureau_balance["SK_ID_BUREAU"].nunique()
)

Bureau Balance shape: (27299925, 3)
Unique bureau loans: 817395


In [62]:
bureau_balance["STATUS"] = bureau_balance["STATUS"].astype(str)

bureau_balance["BB_DPD_FLAG"] = bureau_balance["STATUS"].isin(
    ["1", "2", "3", "4", "5"]
).astype(int)

bureau_balance["BB_SEVERE_DPD_FLAG"] = bureau_balance["STATUS"].isin(
    ["3", "4", "5"]
).astype(int)

bureau_balance["BB_UNKNOWN_FLAG"] = (
    bureau_balance["STATUS"] == "X"
).astype(int)

print("Delinquency indicators created.")

Delinquency indicators created.


In [63]:
bureau_balance_agg = bureau_balance.groupby(
    "SK_ID_BUREAU"
).agg(
    BB_MONTHS_COUNT=("MONTHS_BALANCE", "count"),
    
    BB_DPD_MONTHS=("BB_DPD_FLAG", "sum"),
    
    BB_SEVERE_DPD_MONTHS=("BB_SEVERE_DPD_FLAG", "sum"),
    
    BB_UNKNOWN_MONTHS=("BB_UNKNOWN_FLAG", "sum"),
    
    BB_LAST_MONTH=("MONTHS_BALANCE", "max"),
    
    BB_HISTORY_LENGTH=("MONTHS_BALANCE", "min")
).reset_index()

print("Bureau-loan aggregates:", bureau_balance_agg.shape)
bureau_balance_agg.head()

Bureau-loan aggregates: (817395, 7)


,SK_ID_BUREAU,BB_MONTHS_COUNT,BB_DPD_MONTHS,BB_SEVERE_DPD_MONTHS,BB_UNKNOWN_MONTHS,BB_LAST_MONTH,BB_HISTORY_LENGTH
0,5001709,97,0,0,11,0,-96
1,5001710,83,0,0,30,0,-82
2,5001711,4,0,0,1,0,-3
3,5001712,19,0,0,0,0,-18
4,5001713,22,0,0,22,0,-21


In [64]:
bureau_balance_agg = bureau_balance_agg.merge(
    bureau[["SK_ID_BUREAU", "SK_ID_CURR"]],
    on="SK_ID_BUREAU",
    how="left"
)

bureau_balance_curr = bureau_balance_agg.groupby(
    "SK_ID_CURR"
).agg(
    BB_TOTAL_MONTHS=("BB_MONTHS_COUNT", "sum"),
    
    BB_DPD_MONTHS=("BB_DPD_MONTHS", "sum"),
    
    BB_SEVERE_DPD_MONTHS=("BB_SEVERE_DPD_MONTHS", "sum"),
    
    BB_UNKNOWN_MONTHS=("BB_UNKNOWN_MONTHS", "sum"),
    
    BB_AVG_HISTORY_LENGTH=("BB_MONTHS_COUNT", "mean"),
    
    BB_MAX_HISTORY_LENGTH=("BB_MONTHS_COUNT", "max")
).reset_index()

print(
    "Applicant-level Bureau Balance features:",
    bureau_balance_curr.shape
)

Applicant-level Bureau Balance features: (134542, 7)


In [65]:
bureau_balance_curr["BB_DPD_RATE"] = (
    bureau_balance_curr["BB_DPD_MONTHS"] /
    bureau_balance_curr["BB_TOTAL_MONTHS"].replace(0, np.nan)
)

bureau_balance_curr["BB_SEVERE_DPD_RATE"] = (
    bureau_balance_curr["BB_SEVERE_DPD_MONTHS"] /
    bureau_balance_curr["BB_TOTAL_MONTHS"].replace(0, np.nan)
)

print("Bureau Balance risk features created.")

Bureau Balance risk features created.


In [66]:
df = df.merge(
    bureau_balance_curr,
    on="SK_ID_CURR",
    how="left"
)

print("After Bureau Balance merge:", df.shape)

After Bureau Balance merge: (307511, 154)


In [67]:
previous = pd.read_csv(
    "../data/previous_application.csv"
)

print("Previous Applications shape:", previous.shape)
print(
    "Unique applicants:",
    previous["SK_ID_CURR"].nunique()
)

Previous Applications shape: (1670214, 37)
Unique applicants: 338857


In [68]:
previous["PREV_APPROVED"] = (
    previous["NAME_CONTRACT_STATUS"] == "Approved"
).astype(int)

previous["PREV_REFUSED"] = (
    previous["NAME_CONTRACT_STATUS"] == "Refused"
).astype(int)

previous["PREV_CANCELED"] = (
    previous["NAME_CONTRACT_STATUS"] == "Canceled"
).astype(int)

previous["PREV_UNUSED"] = (
    previous["NAME_CONTRACT_STATUS"] == "Unused offer"
).astype(int)

print("Previous application status indicators created.")

Previous application status indicators created.


In [69]:
previous["PREV_CREDIT_APPLICATION_RATIO"] = (
    previous["AMT_CREDIT"] /
    previous["AMT_APPLICATION"].replace(0, np.nan)
)

previous["PREV_DOWN_PAYMENT_RATIO"] = (
    previous["AMT_DOWN_PAYMENT"] /
    previous["AMT_APPLICATION"].replace(0, np.nan)
)

previous["PREV_CREDIT_GOODS_RATIO"] = (
    previous["AMT_CREDIT"] /
    previous["AMT_GOODS_PRICE"].replace(0, np.nan)
)

print("Previous application ratios created.")

Previous application ratios created.


In [70]:
previous_agg = previous.groupby("SK_ID_CURR").agg(
    PREV_APP_COUNT=("SK_ID_PREV", "count"),
    
    PREV_APPROVED_COUNT=("PREV_APPROVED", "sum"),
    PREV_REFUSED_COUNT=("PREV_REFUSED", "sum"),
    PREV_CANCELED_COUNT=("PREV_CANCELED", "sum"),
    PREV_UNUSED_COUNT=("PREV_UNUSED", "sum"),
    
    PREV_CREDIT_SUM=("AMT_CREDIT", "sum"),
    PREV_CREDIT_MEAN=("AMT_CREDIT", "mean"),
    PREV_CREDIT_MAX=("AMT_CREDIT", "max"),
    
    PREV_APPLICATION_SUM=("AMT_APPLICATION", "sum"),
    PREV_APPLICATION_MEAN=("AMT_APPLICATION", "mean"),
    
    PREV_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
    PREV_DOWN_PAYMENT_MEAN=("AMT_DOWN_PAYMENT", "mean"),
    
    PREV_CREDIT_APPLICATION_RATIO_MEAN=(
        "PREV_CREDIT_APPLICATION_RATIO",
        "mean"
    ),
    
    PREV_DOWN_PAYMENT_RATIO_MEAN=(
        "PREV_DOWN_PAYMENT_RATIO",
        "mean"
    ),
    
    PREV_CREDIT_GOODS_RATIO_MEAN=(
        "PREV_CREDIT_GOODS_RATIO",
        "mean"
    ),
    
    PREV_DAYS_DECISION_MEAN=("DAYS_DECISION", "mean"),
    PREV_DAYS_DECISION_MIN=("DAYS_DECISION", "min"),
    
    PREV_INSTALLMENTS_MEAN=("CNT_PAYMENT", "mean")
).reset_index()

print("Previous application aggregates:", previous_agg.shape)
previous_agg.head()

Previous application aggregates: (338857, 19)


,SK_ID_CURR,PREV_APP_COUNT,PREV_APPROVED_COUNT,PREV_REFUSED_COUNT,PREV_CANCELED_COUNT,PREV_UNUSED_COUNT,PREV_CREDIT_SUM,PREV_CREDIT_MEAN,PREV_CREDIT_MAX,PREV_APPLICATION_SUM,PREV_APPLICATION_MEAN,PREV_ANNUITY_MEAN,PREV_DOWN_PAYMENT_MEAN,PREV_CREDIT_APPLICATION_RATIO_MEAN,PREV_DOWN_PAYMENT_RATIO_MEAN,PREV_CREDIT_GOODS_RATIO_MEAN,PREV_DAYS_DECISION_MEAN,PREV_DAYS_DECISION_MIN,PREV_INSTALLMENTS_MEAN
0,100001,1,1,0,0,0,23787.0,23787.00,23787.0,24835.5,24835.50,3951.000,2520.0,0.957782,0.101468,0.957782,-1740.0,-1740,8.0
1,100002,1,1,0,0,0,179055.0,179055.00,179055.0,179055.0,179055.00,9251.775,0.0,1.000000,0.000000,1.000000,-606.0,-606,24.0
2,100003,3,3,0,0,0,1452573.0,484191.00,1035882.0,1306309.5,435436.50,56553.990,3442.5,1.057664,0.050029,1.057664,-1305.0,-2341,10.0
3,100004,1,1,0,0,0,20106.0,20106.00,20106.0,24282.0,24282.00,5357.250,4860.0,0.828021,0.200148,0.828021,-815.0,-815,4.0
4,100005,2,1,0,1,0,40153.5,20076.75,40153.5,44617.5,22308.75,4813.200,4464.0,0.899950,0.100050,0.899950,-536.0,-757,12.0


In [71]:
previous_agg["PREV_APPROVAL_RATE"] = (
    previous_agg["PREV_APPROVED_COUNT"] /
    previous_agg["PREV_APP_COUNT"]
)

previous_agg["PREV_REFUSAL_RATE"] = (
    previous_agg["PREV_REFUSED_COUNT"] /
    previous_agg["PREV_APP_COUNT"]
)

print("Approval and refusal rates created.")

Approval and refusal rates created.


In [72]:
df = df.merge(
    previous_agg,
    on="SK_ID_CURR",
    how="left"
)

print("After Previous Applications merge:", df.shape)

After Previous Applications merge: (307511, 174)


In [73]:
installments = pd.read_csv(
    "../data/installments_payments.csv"
)

print("Installments shape:", installments.shape)
print(
    "Unique applicants:",
    installments["SK_ID_CURR"].nunique()
)

Installments shape: (13605401, 8)
Unique applicants: 339587


In [74]:
installments["PAYMENT_DELAY"] = (
    installments["DAYS_ENTRY_PAYMENT"]
    - installments["DAYS_INSTALMENT"]
)

installments["PAYMENT_DIFF"] = (
    installments["AMT_PAYMENT"]
    - installments["AMT_INSTALMENT"]
)

installments["LATE_PAYMENT"] = (
    installments["PAYMENT_DELAY"] > 0
).astype(int)

installments["UNDERPAYMENT"] = (
    installments["PAYMENT_DIFF"] < 0
).astype(int)

installments["PAYMENT_RATIO"] = (
    installments["AMT_PAYMENT"]
    / installments["AMT_INSTALMENT"].replace(0, np.nan)
)

print("Installment behavior features created.")

Installment behavior features created.


In [75]:
installments_agg = installments.groupby(
    "SK_ID_CURR"
).agg(
    INSTALLMENT_COUNT=("SK_ID_PREV", "count"),

    PAYMENT_DELAY_MEAN=("PAYMENT_DELAY", "mean"),
    PAYMENT_DELAY_MAX=("PAYMENT_DELAY", "max"),

    LATE_PAYMENT_COUNT=("LATE_PAYMENT", "sum"),
    UNDERPAYMENT_COUNT=("UNDERPAYMENT", "sum"),

    PAYMENT_DIFF_MEAN=("PAYMENT_DIFF", "mean"),
    PAYMENT_DIFF_MIN=("PAYMENT_DIFF", "min"),

    PAYMENT_RATIO_MEAN=("PAYMENT_RATIO", "mean"),
    PAYMENT_RATIO_MIN=("PAYMENT_RATIO", "min"),

    AMT_INSTALMENT_SUM=("AMT_INSTALMENT", "sum"),
    AMT_PAYMENT_SUM=("AMT_PAYMENT", "sum")
).reset_index()

print(
    "Installment aggregates:",
    installments_agg.shape
)

Installment aggregates: (339587, 12)


In [76]:
installments_agg["LATE_PAYMENT_RATE"] = (
    installments_agg["LATE_PAYMENT_COUNT"]
    / installments_agg["INSTALLMENT_COUNT"]
)

installments_agg["UNDERPAYMENT_RATE"] = (
    installments_agg["UNDERPAYMENT_COUNT"]
    / installments_agg["INSTALLMENT_COUNT"]
)

installments_agg["TOTAL_PAYMENT_RATIO"] = (
    installments_agg["AMT_PAYMENT_SUM"]
    / installments_agg["AMT_INSTALMENT_SUM"].replace(0, np.nan)
)

print("Repayment rate features created.")

Repayment rate features created.


In [77]:
df = df.merge(
    installments_agg,
    on="SK_ID_CURR",
    how="left"
)

print("After Installments merge:", df.shape)

After Installments merge: (307511, 188)


In [78]:
pos_cash = pd.read_csv(
    "../data/POS_CASH_balance.csv"
)

print("POS Cash shape:", pos_cash.shape)
print(
    "Unique applicants:",
    pos_cash["SK_ID_CURR"].nunique()
)

POS Cash shape: (10001358, 8)
Unique applicants: 337252


In [79]:
pos_cash["POS_DPD_FLAG"] = (
    pos_cash["SK_DPD"] > 0
).astype(int)

pos_cash["POS_DPD_SEVERE_FLAG"] = (
    pos_cash["SK_DPD"] >= 30
).astype(int)

pos_cash["POS_DPD_DEF_FLAG"] = (
    pos_cash["SK_DPD_DEF"] > 0
).astype(int)

print("POS delinquency indicators created.")

POS delinquency indicators created.


In [80]:
pos_cash_agg = pos_cash.groupby(
    "SK_ID_CURR"
).agg(
    POS_MONTH_COUNT=("MONTHS_BALANCE", "count"),

    POS_DPD_MEAN=("SK_DPD", "mean"),
    POS_DPD_MAX=("SK_DPD", "max"),

    POS_DPD_COUNT=("POS_DPD_FLAG", "sum"),
    POS_SEVERE_DPD_COUNT=("POS_DPD_SEVERE_FLAG", "sum"),
    POS_DPD_DEF_COUNT=("POS_DPD_DEF_FLAG", "sum"),

    POS_INSTALMENT_MEAN=("CNT_INSTALMENT", "mean"),
    POS_INSTALMENT_FUTURE_MEAN=(
        "CNT_INSTALMENT_FUTURE",
        "mean"
    ),

    POS_INSTALMENT_FUTURE_MAX=(
        "CNT_INSTALMENT_FUTURE",
        "max"
    )
).reset_index()

print(
    "POS Cash aggregates:",
    pos_cash_agg.shape
)

POS Cash aggregates: (337252, 10)


In [81]:
pos_cash_agg["POS_DPD_RATE"] = (
    pos_cash_agg["POS_DPD_COUNT"]
    / pos_cash_agg["POS_MONTH_COUNT"]
)

pos_cash_agg["POS_SEVERE_DPD_RATE"] = (
    pos_cash_agg["POS_SEVERE_DPD_COUNT"]
    / pos_cash_agg["POS_MONTH_COUNT"]
)

pos_cash_agg["POS_DPD_DEF_RATE"] = (
    pos_cash_agg["POS_DPD_DEF_COUNT"]
    / pos_cash_agg["POS_MONTH_COUNT"]
)

print("POS risk rates created.")

POS risk rates created.


In [82]:
df = df.merge(
    pos_cash_agg,
    on="SK_ID_CURR",
    how="left"
)

print("After POS Cash merge:", df.shape)

After POS Cash merge: (307511, 200)


In [83]:
credit_card = pd.read_csv(
    "../data/credit_card_balance.csv"
)

print("Credit Card shape:", credit_card.shape)
print(
    "Unique applicants:",
    credit_card["SK_ID_CURR"].nunique()
)

Credit Card shape: (3840312, 23)
Unique applicants: 103558


In [84]:
credit_card["CC_DPD_FLAG"] = (
    credit_card["SK_DPD"] > 0
).astype(int)

credit_card["CC_SEVERE_DPD_FLAG"] = (
    credit_card["SK_DPD"] >= 30
).astype(int)

credit_card["CC_DPD_DEF_FLAG"] = (
    credit_card["SK_DPD_DEF"] > 0
).astype(int)

print("Credit card delinquency indicators created.")

Credit card delinquency indicators created.


In [85]:
credit_card["CC_UTILIZATION"] = (
    credit_card["AMT_BALANCE"] /
    credit_card["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
)

credit_card["CC_PAYMENT_RATIO"] = (
    credit_card["AMT_PAYMENT_CURRENT"] /
    credit_card["AMT_INST_MIN_REGULARITY"].replace(0, np.nan)
)

print("Credit card utilization features created.")

Credit card utilization features created.


In [86]:
credit_card_agg = credit_card.groupby(
    "SK_ID_CURR"
).agg(
    CC_MONTH_COUNT=("MONTHS_BALANCE", "count"),

    CC_BALANCE_MEAN=("AMT_BALANCE", "mean"),
    CC_BALANCE_MAX=("AMT_BALANCE", "max"),

    CC_CREDIT_LIMIT_MEAN=(
        "AMT_CREDIT_LIMIT_ACTUAL",
        "mean"
    ),

    CC_UTILIZATION_MEAN=("CC_UTILIZATION", "mean"),
    CC_UTILIZATION_MAX=("CC_UTILIZATION", "max"),

    CC_DPD_MEAN=("SK_DPD", "mean"),
    CC_DPD_MAX=("SK_DPD", "max"),

    CC_DPD_COUNT=("CC_DPD_FLAG", "sum"),
    CC_SEVERE_DPD_COUNT=("CC_SEVERE_DPD_FLAG", "sum"),
    CC_DPD_DEF_COUNT=("CC_DPD_DEF_FLAG", "sum"),

    CC_PAYMENT_MEAN=("AMT_PAYMENT_CURRENT", "mean"),
    CC_PAYMENT_MAX=("AMT_PAYMENT_CURRENT", "max"),

    CC_PAYMENT_RATIO_MEAN=("CC_PAYMENT_RATIO", "mean")
).reset_index()

print(
    "Credit Card aggregates:",
    credit_card_agg.shape
)

Credit Card aggregates: (103558, 15)


In [87]:
credit_card_agg["CC_DPD_RATE"] = (
    credit_card_agg["CC_DPD_COUNT"]
    / credit_card_agg["CC_MONTH_COUNT"]
)

credit_card_agg["CC_SEVERE_DPD_RATE"] = (
    credit_card_agg["CC_SEVERE_DPD_COUNT"]
    / credit_card_agg["CC_MONTH_COUNT"]
)

credit_card_agg["CC_DPD_DEF_RATE"] = (
    credit_card_agg["CC_DPD_DEF_COUNT"]
    / credit_card_agg["CC_MONTH_COUNT"]
)

print("Credit card risk rates created.")

Credit card risk rates created.


In [88]:
df = df.merge(
    credit_card_agg,
    on="SK_ID_CURR",
    how="left"
)

print("After Credit Card merge:", df.shape)

After Credit Card merge: (307511, 217)


In [89]:
print("Final dataset shape:", df.shape)

print("\nDuplicate columns:")
print(df.columns[df.columns.duplicated()].tolist())

print("\nDuplicate column count:",
      df.columns.duplicated().sum())

print("\nUnique applicants:",
      df["SK_ID_CURR"].nunique())

print("\nTarget distribution:")
print(df["TARGET"].value_counts())

Final dataset shape: (307511, 217)

Duplicate columns:
[]

Duplicate column count: 0

Unique applicants: 307511

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64


In [90]:
missing_summary = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

missing_summary = missing_summary[
    missing_summary > 0
]

print("Features with missing values:",
      len(missing_summary))

print(missing_summary.head(30))

Features with missing values: 161
CC_PAYMENT_RATIO_MEAN       248242
CC_PAYMENT_MAX              246451
CC_PAYMENT_MEAN             246451
CC_UTILIZATION_MEAN         221475
CC_UTILIZATION_MAX          221475
CC_DPD_DEF_RATE             220606
CC_DPD_COUNT                220606
CC_BALANCE_MAX              220606
CC_MONTH_COUNT              220606
CC_CREDIT_LIMIT_MEAN        220606
CC_DPD_MEAN                 220606
CC_DPD_MAX                  220606
CC_BALANCE_MEAN             220606
CC_SEVERE_DPD_COUNT         220606
CC_DPD_DEF_COUNT            220606
CC_DPD_RATE                 220606
CC_SEVERE_DPD_RATE          220606
BB_SEVERE_DPD_MONTHS        215280
BB_UNKNOWN_MONTHS           215280
BB_TOTAL_MONTHS             215280
BB_DPD_MONTHS               215280
BB_AVG_HISTORY_LENGTH       215280
BB_MAX_HISTORY_LENGTH       215280
BB_DPD_RATE                 215280
BB_SEVERE_DPD_RATE          215280
COMMONAREA_AVG              214865
COMMONAREA_MEDI             214865
COMMONAREA_MODE      

In [91]:
historical_features = [
    col for col in df.columns
    if col.startswith(("BUREAU_", "BB_", "PREV_", "INSTALLMENT_", "PAYMENT_",
                       "POS_", "CC_"))
]

print("Historical features:", len(historical_features))

print("\nHistorical feature names:")
print(historical_features)

Historical features: 83

Historical feature names:
['BUREAU_LOAN_COUNT', 'BUREAU_ACTIVE_COUNT', 'BUREAU_CLOSED_COUNT', 'BUREAU_CREDIT_SUM', 'BUREAU_CREDIT_MEAN', 'BUREAU_CREDIT_MAX', 'BUREAU_DEBT_SUM', 'BUREAU_DEBT_MEAN', 'BUREAU_OVERDUE_SUM', 'BUREAU_OVERDUE_MAX', 'BUREAU_DPD_MEAN', 'BUREAU_DPD_MAX', 'BUREAU_PROLONG_COUNT', 'BUREAU_DAYS_CREDIT_MEAN', 'BUREAU_DAYS_CREDIT_MIN', 'BUREAU_CREDIT_TYPE_COUNT', 'BUREAU_DEBT_CREDIT_RATIO', 'BUREAU_ACTIVE_RATIO', 'BUREAU_OVERDUE_LOAN_RATIO', 'BB_TOTAL_MONTHS', 'BB_DPD_MONTHS', 'BB_SEVERE_DPD_MONTHS', 'BB_UNKNOWN_MONTHS', 'BB_AVG_HISTORY_LENGTH', 'BB_MAX_HISTORY_LENGTH', 'BB_DPD_RATE', 'BB_SEVERE_DPD_RATE', 'PREV_APP_COUNT', 'PREV_APPROVED_COUNT', 'PREV_REFUSED_COUNT', 'PREV_CANCELED_COUNT', 'PREV_UNUSED_COUNT', 'PREV_CREDIT_SUM', 'PREV_CREDIT_MEAN', 'PREV_CREDIT_MAX', 'PREV_APPLICATION_SUM', 'PREV_APPLICATION_MEAN', 'PREV_ANNUITY_MEAN', 'PREV_DOWN_PAYMENT_MEAN', 'PREV_CREDIT_APPLICATION_RATIO_MEAN', 'PREV_DOWN_PAYMENT_RATIO_MEAN', 'PREV_CREDIT_

In [92]:
numeric_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns

inf_counts = np.isinf(
    df[numeric_cols]
).sum()

inf_counts = inf_counts[inf_counts > 0]

print("Features containing infinity:", len(inf_counts))
print(inf_counts)

Features containing infinity: 0
Series([], dtype: int64)


In [93]:
historical_count_features = [
    col for col in historical_features
    if any(word in col for word in [
        "COUNT",
        "COUNTS",
        "LOAN_COUNT",
        "APP_COUNT",
        "MONTH_COUNT",
        "DPD_COUNT",
        "MONTHS",
        "INSTALLMENT_COUNT"
    ])
]

historical_rate_features = [
    col for col in historical_features
    if "RATE" in col
]

print("Historical count features:")
print(historical_count_features)

print("\nHistorical rate features:")
print(historical_rate_features)

Historical count features:
['BUREAU_LOAN_COUNT', 'BUREAU_ACTIVE_COUNT', 'BUREAU_CLOSED_COUNT', 'BUREAU_PROLONG_COUNT', 'BUREAU_CREDIT_TYPE_COUNT', 'BB_TOTAL_MONTHS', 'BB_DPD_MONTHS', 'BB_SEVERE_DPD_MONTHS', 'BB_UNKNOWN_MONTHS', 'PREV_APP_COUNT', 'PREV_APPROVED_COUNT', 'PREV_REFUSED_COUNT', 'PREV_CANCELED_COUNT', 'PREV_UNUSED_COUNT', 'INSTALLMENT_COUNT', 'POS_MONTH_COUNT', 'POS_DPD_COUNT', 'POS_SEVERE_DPD_COUNT', 'POS_DPD_DEF_COUNT', 'CC_MONTH_COUNT', 'CC_DPD_COUNT', 'CC_SEVERE_DPD_COUNT', 'CC_DPD_DEF_COUNT']

Historical rate features:
['BB_DPD_RATE', 'BB_SEVERE_DPD_RATE', 'PREV_APPROVAL_RATE', 'PREV_REFUSAL_RATE', 'POS_DPD_RATE', 'POS_SEVERE_DPD_RATE', 'POS_DPD_DEF_RATE', 'CC_DPD_RATE', 'CC_SEVERE_DPD_RATE', 'CC_DPD_DEF_RATE']


In [94]:
historical_missing = (
    df[historical_features]
    .isnull()
    .mean()
    .sort_values(ascending=False)
)

print("Historical features with highest missing rates:")
print((historical_missing * 100).head(30))

Historical features with highest missing rates:
CC_PAYMENT_RATIO_MEAN        80.726218
CC_PAYMENT_MAX               80.143800
CC_PAYMENT_MEAN              80.143800
CC_UTILIZATION_MAX           72.021814
CC_UTILIZATION_MEAN          72.021814
CC_DPD_COUNT                 71.739222
CC_BALANCE_MEAN              71.739222
CC_BALANCE_MAX               71.739222
CC_CREDIT_LIMIT_MEAN         71.739222
CC_DPD_MEAN                  71.739222
CC_DPD_MAX                   71.739222
CC_DPD_DEF_RATE              71.739222
CC_SEVERE_DPD_COUNT          71.739222
CC_DPD_DEF_COUNT             71.739222
CC_SEVERE_DPD_RATE           71.739222
CC_DPD_RATE                  71.739222
CC_MONTH_COUNT               71.739222
BB_TOTAL_MONTHS              70.007252
BB_DPD_MONTHS                70.007252
BB_UNKNOWN_MONTHS            70.007252
BB_AVG_HISTORY_LENGTH        70.007252
BB_MAX_HISTORY_LENGTH        70.007252
BB_DPD_RATE                  70.007252
BB_SEVERE_DPD_RATE           70.007252
BB_SEVERE_DPD_MO

In [95]:
historical_missing_features = pd.DataFrame(
    {
        f"{col}_MISSING": df[col].isna().astype(int)
        for col in historical_features
    },
    index=df.index
)

df = pd.concat(
    [df, historical_missing_features],
    axis=1
)

print(
    "Historical missingness indicators added:",
    historical_missing_features.shape[1]
)

print("New dataset shape:", df.shape)

Historical missingness indicators added: 83
New dataset shape: (307511, 300)


In [96]:
print("Final feature count:", df.shape[1] - 2)
print("Final dataset shape:", df.shape)

Final feature count: 298
Final dataset shape: (307511, 300)


In [97]:
X_full = df.drop(columns=["TARGET"])
y_full = df["TARGET"]

print("X shape:", X_full.shape)
print("y shape:", y_full.shape)

X shape: (307511, 299)
y shape: (307511,)


In [98]:
X_full = X_full.drop(columns=["SK_ID_CURR"])

print("Final modeling features:", X_full.shape[1])

Final modeling features: 298


In [99]:
from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(
    X_full,
    y_full,
    test_size=0.40,
    stratify=y_full,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nTarget distributions:")
print("Train:", y_train.value_counts().to_dict())
print("Validation:", y_val.value_counts().to_dict())
print("Test:", y_test.value_counts().to_dict())

Train: (184506, 298)
Validation: (61502, 298)
Test: (61503, 298)

Target distributions:
Train: {0: 169611, 1: 14895}
Validation: {0: 56537, 1: 4965}
Test: {0: 56538, 1: 4965}


In [100]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Numerical features: 282
Categorical features: 16
Total features: 298


In [101]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Final preprocessing pipeline created!")

Final preprocessing pipeline created!


In [102]:
# Remove duplicated column names
X_train = X_train.loc[:, ~X_train.columns.duplicated()]
X_val = X_val.loc[:, ~X_val.columns.duplicated()]
X_test = X_test.loc[:, ~X_test.columns.duplicated()]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("Duplicate columns:",
      X_train.columns.duplicated().sum())

Train: (184506, 298)
Validation: (61502, 298)
Test: (61503, 298)
Duplicate columns: 0


In [103]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Numerical features: 282
Categorical features: 16
Total features: 298


In [104]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Final preprocessing pipeline created!")

Final preprocessing pipeline created!


In [105]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

print("Final preprocessing completed!")

print("Train processed:", X_train_processed.shape)
print("Validation processed:", X_val_processed.shape)
print("Test processed:", X_test_processed.shape)

Final preprocessing completed!
Train processed: (184506, 422)
Validation processed: (61502, 422)
Test processed: (61503, 422)


In [106]:
# Save final processed feature names
final_feature_names = preprocessor.get_feature_names_out()

joblib.dump(
    final_feature_names,
    "../data/final_feature_names.pkl"
)

print("Feature names saved:", len(final_feature_names))

NameError: name 'joblib' is not defined

In [116]:
print("Training missing values:",
      np.isnan(X_train_processed).sum())

print("Validation missing values:",
      np.isnan(X_val_processed).sum())

print("Test missing values:",
      np.isnan(X_test_processed).sum())

Training missing values: 0
Validation missing values: 0
Test missing values: 0


In [117]:
negative_samples = (y_train == 0).sum()
positive_samples = (y_train == 1).sum()

scale_pos_weight = negative_samples / positive_samples

print("Negative samples:", negative_samples)
print("Positive samples:", positive_samples)
print("Scale Pos Weight:", scale_pos_weight)

Negative samples: 169611
Positive samples: 14895
Scale Pos Weight: 11.38710976837865


In [118]:
import joblib

joblib.dump(
    X_train_processed,
    "../data/X_train_final_processed.pkl"
)

joblib.dump(
    X_val_processed,
    "../data/X_val_final_processed.pkl"
)

joblib.dump(
    X_test_processed,
    "../data/X_test_final_processed.pkl"
)

joblib.dump(
    y_train,
    "../data/y_train_final.pkl"
)

joblib.dump(
    y_val,
    "../data/y_val_final.pkl"
)

joblib.dump(
    y_test,
    "../data/y_test_final.pkl"
)

print("FINAL 422-feature datasets saved successfully!")

FINAL 422-feature datasets saved successfully!
